In [1]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
import gensim.downloader as downloader
import pandas as pd
import numpy as np

In [ ]:
# Load both csv files into separate dataframes
# Source: https://www.kaggle.com/datasets/emineyetm/fake-news-detection-datasets/data
true_data = pd.read_csv('News _dataset/True.csv')
fake_data = pd.read_csv('News _dataset/Fake.csv')

# Removing true data signature: 'LOCATION (REUTERS) - '
def remove_signature(text: str):
  try:
    texts = text.split('-')
    return ' '.join(texts[1:])
  except Exception:
    return text

# Remove the signatures
true_data['text'] = true_data['text'].apply(remove_signature)

# Create a new target feature
true_data['fakeNews'] = 0
fake_data['fakeNews'] = 1

# Merge the two dfs into one
data = pd.concat([true_data, fake_data], ignore_index=True)
# No duplicates or NaN to be dropped
data.drop_duplicates()
data.dropna()

,title,text,subject,date,fakeNews
0,"As U.S. budget fight looms, Republicans flip t...",The head of a conservative Republican faction...,politicsNews,"December 31, 2017",0
1,U.S. military to accept transgender recruits o...,Transgender people will be allowed for the fi...,politicsNews,"December 29, 2017",0
2,Senior U.S. Republican senator: 'Let Mr. Muell...,The special counsel investigation of links be...,politicsNews,"December 31, 2017",0
3,FBI Russia probe helped by Australian diplomat...,Trump campaign adviser George Papadopoulos to...,politicsNews,"December 30, 2017",0
4,Trump wants Postal Service to charge 'much mor...,President Donald Trump called on the U.S. Pos...,politicsNews,"December 29, 2017",0
...,...,...,...,...,...
44893,McPain: John McCain Furious That Iran Treated ...,21st Century Wire says As 21WIRE reported earl...,Middle-east,"January 16, 2016",1
44894,JUSTICE? Yahoo Settles E-mail Privacy Class-ac...,21st Century Wire says It s a familiar theme. ...,Middle-east,"January 16, 2016",1
44895,Sunnistan: US and Allied ‘Safe Zone’ Plan to T...,Patrick Henningsen 21st Century WireRemember ...,Middle-east,"January 15, 2016",1
44896,How to Blow $700 Million: Al Jazeera America F...,21st Century Wire says Al Jazeera America will...,Middle-east,"January 14, 2016",1


In [ ]:
import re
import string

# Data Preprocessing
def preprocess_text(text: str):
  # From: https://www.trantorinc.com/blog/natural-language-processing-with-python
  text = text.lower()
  text = re.sub(r'http[s]?://\S+|www\.\S+', '', text) # Remove links
  text = re.sub(r'<.*?>', ' ', text) # HTML tags
  text = text.translate(str.maketrans(' ', ' ', string.punctuation + '“”')) # remove punctuation
  text = ' '.join(text.split()) # Deal with extra whitespace
  return text

def date_to_numeric(date: str):
  # 'month day, year '
  try:
    date = date.replace(',', '')
    date = date.strip().split()
    months = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
    return (int(date[2])-2015)*365 + months.index(date[0])*31 + int(date[1])
  except Exception:
    return 0

# Process the titles and text
data['title'] = data['title'].apply(preprocess_text)
data['text'] = data['text'].apply(preprocess_text)

# Mapping unique integer for each unique subject
mapping = {label:idx for idx,label in enumerate(data['subject'].unique())}
data['subject'] = data['subject'].map(mapping)

# Process the dates
data['date'] = data['date'].apply(date_to_numeric)

In [ ]:
# Training Doc2Vec model on dataset
# To skip the training, uncomment the cell below to load the trained model
tagged_data = [TaggedDocument(words=doc.split(), tags=[str(i)]) for i, doc in enumerate(data['text'].to_list())]
d2v = Doc2Vec(vector_size=250, window=7, workers=8, epochs=10, min_count=10)
d2v.build_vocab(tagged_data)
d2v.train(tagged_data, total_examples=d2v.corpus_count, epochs=d2v.epochs)

In [ ]:
# Uncomment to load the Doc2Vec model without having to train
# d2v = Doc2Vec.load('d2v_news_250.model')

In [ ]:
# Doc2Vec on texts
def get_doc2vec_vector(article: str, model):
  return model.infer_vector(article.split())

# Applying the Doc2Vec model on texts
data['text'] = data['text'].apply(lambda x: get_doc2vec_vector(x, d2v))

In [13]:
# Google's Word2Vec model
w2v = downloader.load("word2vec-google-news-300")

[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [ ]:
def get_avg_word2vec_vector(title: str, model):
  # Tokenize the title into words
  words = title.split()
  word_vectors = [model[word] for word in words if word in model.key_to_index] # Get word vectors
  if not word_vectors:
    return np.zeros(model.vector_size)
  # average of all word vectors
  return np.mean(word_vectors, axis=0)

# Applying Word2Vec model on titles
data['title'] = data['title'].apply(lambda x: get_avg_word2vec_vector(x, w2v))

0        [0.11317444, 0.05189514, 0.009300232, 0.167095...
1        [-0.09843445, 0.070007324, 0.09357643, 0.16351...
2        [0.07041422, 0.028238932, 0.026701186, 0.13220...
3        [-0.035939533, 0.05050998, 0.05574544, 0.10226...
4        [0.049884032, 0.050390624, 0.007580566, 0.0960...
                               ...                        
44893    [-0.025634766, 0.10160404, 0.05184767, 0.14841...
44894    [0.022562662, 0.049540203, -0.017201742, 0.086...
44895    [0.0064426, 0.008748372, 0.053265043, 0.066162...
44896    [0.047471788, 0.054261737, 0.009941949, 0.0659...
44897    [-0.005193537, 0.07102273, 0.021114003, 0.1741...
Name: title, Length: 44898, dtype: object


In [ ]:
# Convert title and text vectors from lists to independent features
title_expanded = pd.DataFrame(data['title'].tolist())
text_expanded = pd.DataFrame(data['text'].tolist())

# Drop title and text list features
data = data.drop(['title', 'text'], axis=1)
data = pd.concat([data, title_expanded, text_expanded], axis=1)

In [ ]:
# Optional code to download dataframe to csv if not already downloaded
# data.to_csv('fake_news_data.csv')